# MobileMonoDETR-Student-A2 GT-only baseline

This is the accuracy-first Vehicle/Pedestrian baseline. It changes only the frozen R0 ResNet50 backbone and required feature projections to MobileNetV4 Conv Medium. Distillation is disabled. Run top-to-bottom on a GPU runtime. Checkpoints, combined logs, manifests, and sweep outputs are durable on Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, re, shlex, shutil, subprocess, sys
MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
MONODETR_KITTI = Path('/content/monodetr_kitti_a2')
R0_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_gt')
RUN_NAME = 'monodetr_a2_mnv4_vehicle_pedestrian_gt'
MAX_EPOCHS = 195
SAVE_FREQUENCY = 5
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
def run(command, cwd=None, env=None):
    command = [str(x) for x in command]; print('+', shlex.join(command), flush=True)
    merged = os.environ.copy(); merged.update(env or {})
    result = subprocess.run(command, cwd=cwd, env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])

In [ ]:
# Fetch pinned MonoDETR and install all required compatibility/model patches.
if not MOBILE_REPO.exists(): run(['git', 'clone', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO])
else: run(['git', 'pull', '--ff-only'], cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run(['git', 'fetch', '--all'], cwd=MONODETR_REPO)
run(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'tqdm', 'ninja', 'timm==1.0.20', 'pandas'])
A2_BACKBONE = 'mobilenetv4_conv_medium.e500_r256_in1k'
run([sys.executable, '-c', f"import timm; name='{A2_BACKBONE}'; assert timm.__version__ == '1.0.20', timm.__version__; assert name in timm.list_models(pretrained=True), name; m=timm.create_model(name, pretrained=False, features_only=True, out_indices=(2,3,4)); print('A2 backbone:', name, 'channels=', m.feature_info.channels(), 'strides=', m.feature_info.reduction())"])

for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_product_taxonomy.py', 'patch_monodetr_mobilenetv4.py', 'patch_monodetr_verbose_resume.py', 'patch_monodetr_checkpoint_metadata.py'):
    run([sys.executable, f'scripts/{patch}', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
ops = MONODETR_REPO/'lib/models/monodetr/ops'
shutil.rmtree(ops/'build', ignore_errors=True)
run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops, env={'MAX_JOBS': '2'})
run([sys.executable, '-c', 'import torch, timm, MultiScaleDeformableAttention; from lib.models.monodetr import build_monodetr; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))'], cwd=MONODETR_REPO)

In [ ]:
# Create the exact Chen-split KITTI view, preferring an already staged local copy.
def resolve(root, names):
    for name in names:
        path = root/name
        if path.is_dir(): return path
sources = {}
for key, names in {'image_2':['training/image_2','training/image_02'], 'label_2':['training/label_2','training/label_02'], 'calib':['training/calib']}.items():
    sources[key] = resolve(LOCAL_DATASET_ROOT, names) or resolve(DRIVE_DATASET_ROOT, names)
if any(path is None for path in sources.values()): raise FileNotFoundError(f'Missing KITTI sources: {sources}')
(MONODETR_KITTI/'training').mkdir(parents=True, exist_ok=True); (MONODETR_KITTI/'ImageSets').mkdir(parents=True, exist_ok=True)
for name, target in sources.items():
    link = MONODETR_KITTI/'training'/name
    if link.is_symlink() and link.resolve() == target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt', MONODETR_KITTI/'ImageSets'/f'{split}.txt')
assert len((MONODETR_KITTI/'ImageSets/train.txt').read_text().splitlines()) == 3712
assert len((MONODETR_KITTI/'ImageSets/val.txt').read_text().splitlines()) == 3769
print('A2 KITTI view:', MONODETR_KITTI, sources)

In [ ]:
# Fail closed on frozen R0 epoch/hash, then create the A2 initialization/config/manifest.
if not R0_SELECTION.is_file(): raise FileNotFoundError(R0_SELECTION)
run([sys.executable, 'scripts/prepare_monodetr_a2_student.py', '--monodetr-repo', MONODETR_REPO, '--dataset-root', MONODETR_KITTI, '--r0-selection', R0_SELECTION, '--output-root', OUTPUT_ROOT, '--run-name', RUN_NAME, '--max-epochs', MAX_EPOCHS, '--save-frequency', SAVE_FREQUENCY, '--batch-size', BATCH_SIZE, '--learning-rate', LEARNING_RATE], cwd=MOBILE_REPO)
CONFIG = MONODETR_REPO/'configs/monodetr_a2_mnv4_vehicle_pedestrian_gt.yaml'
RUN_DIR = OUTPUT_ROOT/RUN_NAME
print(CONFIG.read_text())
print((RUN_DIR/'experiment_manifest.json').read_text())

## Real GT-only training

The next two cells find only complete epoch checkpoints belonging to this exact A2 run, restore model and optimizer state, stream every batch/epoch line into the notebook, and duplicate stdout/stderr into a timestamped Drive log. Re-run both cells after an interruption. Do not change batch size or schedule inside this run.

In [ ]:
import torch, yaml
checkpoint_pattern = re.compile(r'^checkpoint_epoch_(\d+)\.pth$')
valid_checkpoints = []
for path in RUN_DIR.glob('checkpoint_epoch_*.pth'):
    match = checkpoint_pattern.match(path.name)
    if not match: continue
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
        epoch = int(payload.get('epoch', -1))
        if epoch != int(match.group(1)): raise ValueError('filename/payload epoch mismatch')
        if payload.get('model_state') is None: raise ValueError('model_state missing')
        if payload.get('optimizer_state') is None: raise ValueError('optimizer_state missing')
        valid_checkpoints.append((epoch, path)); print(f'Valid checkpoint: epoch={epoch} size={path.stat().st_size/1e6:.1f}MB {path}')
    except Exception as error: print(f'Skipping invalid checkpoint {path}: {type(error).__name__}: {error}')
latest = max(valid_checkpoints, default=None, key=lambda item: item[0])
run_cfg = yaml.safe_load(CONFIG.read_text())
if latest is None:
    START_EPOCH = 0; CONFIG_TO_RUN = CONFIG
    print('Starting fresh from the frozen A2 initialization.')
else:
    START_EPOCH, RESUME_CHECKPOINT = latest
    run_cfg['trainer'].pop('pretrain_model', None)
    run_cfg['trainer']['resume_model'] = str(RESUME_CHECKPOINT)
    run_cfg['trainer']['max_epoch'] = MAX_EPOCHS
    CONFIG_TO_RUN = MONODETR_REPO/'configs/monodetr_a2_mnv4_vehicle_pedestrian_gt_resume.yaml'
    CONFIG_TO_RUN.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
    print(f'Resuming exact A2 run after epoch {START_EPOCH}: {RESUME_CHECKPOINT}')
print('Training config:', CONFIG_TO_RUN, 'remaining epochs:', max(0, MAX_EPOCHS-START_EPOCH))

In [ ]:
from collections import deque
from datetime import datetime, timezone
LOG_DIR = OUTPUT_ROOT/'colab_logs'; LOG_DIR.mkdir(parents=True, exist_ok=True)
def run_training_logged(command, cwd):
    command = [str(x) for x in command]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    log_path = LOG_DIR/f'train_{RUN_NAME}_{stamp}.log'
    print('+', shlex.join(command), '\nDurable combined log:', log_path, flush=True)
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'; tail = deque(maxlen=120)
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); tail.append(line.rstrip())
        return_code = process.wait()
    if return_code:
        print('Last captured lines:\n' + ('\n'.join(tail) if tail else '<no output>'))
        subprocess.run(['nvidia-smi']); subprocess.run(['df', '-h', '/content', '/content/drive'])
        raise RuntimeError(f'Training exited {return_code}; durable log: {log_path}')
    return log_path
if START_EPOCH >= MAX_EPOCHS: print(f'A2 already reached epoch {START_EPOCH}; no training required.')
else: TRAIN_LOG = run_training_logged([sys.executable, '-u', 'tools/train_val.py', '--config', CONFIG_TO_RUN], MONODETR_REPO)

## Restartable product checkpoint sweep

Run after training. Completed per-epoch evaluations are cached. The report selects by balanced moderate 3D AP_R40 and separately evaluates all five 90%-of-R0 gates. Passing also requires a later nearby-recall review. No distillation or retraining occurs here.

In [ ]:
SWEEP_DIR = OUTPUT_ROOT/'product_checkpoint_sweep'
run([sys.executable, '-u', 'scripts/sweep_monodetr_a2_product_checkpoints.py', '--monodetr-repo', MONODETR_REPO, '--mobile-repo', MOBILE_REPO, '--training-config', CONFIG, '--run-dir', RUN_DIR, '--dataset-root', MONODETR_KITTI, '--split-dir', SPLIT_DIR, '--output-dir', SWEEP_DIR, '--product-config', 'configs/kitti_mobileadas3d_s1.yaml', '--profile', 'colab_drive', '--score-threshold', '0.001', '--topk', '50'], cwd=MOBILE_REPO)
print((SWEEP_DIR/'a2_product_selection.json').read_text())
import pandas as pd
ranking = pd.read_csv(SWEEP_DIR/'r0_product_checkpoint_sweep.csv')
display(ranking[['rank','epoch','vehicle_3d_moderate','pedestrian_3d_moderate','mean_3d_moderate','vehicle_bev_moderate','pedestrian_bev_moderate']].head(15))

## Frozen epoch-130 nearby-recall and geometry diagnostic

Run after the checkpoint sweep. This performs inference only for the frozen epoch-130 checkpoint, verifies its SHA-256, requires all 3,769 prediction files, and reports the product near-field recall gates plus matched Vehicle/Pedestrian geometry errors. It does not train or modify the checkpoint.


In [ ]:
# Recreate predictions for the frozen A2 epoch-130 checkpoint, then run the product diagnostic.
import pandas as pd
import yaml
SELECTED_EPOCH = 130
SELECTED_SHA256 = 'ed2134a98acbf1ab2fc61f7c8749b38fdfd2418e7f7932593e5e37a8d9ef33f4'
SELECTED_CHECKPOINT = RUN_DIR/f'checkpoint_epoch_{SELECTED_EPOCH}.pth'
if not SELECTED_CHECKPOINT.is_file(): raise FileNotFoundError(SELECTED_CHECKPOINT)
import hashlib
def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
actual_hash=sha256(SELECTED_CHECKPOINT)
if actual_hash != SELECTED_SHA256: raise RuntimeError(f'A2 epoch-130 hash mismatch: {actual_hash}')
selection = __import__('json').loads((SWEEP_DIR/'a2_product_selection.json').read_text())
if int(selection['selected_epoch']) != SELECTED_EPOCH: raise RuntimeError(selection)
diag_cfg = yaml.safe_load(CONFIG.read_text())
diag_cfg['tester'].update({'mode':'single','checkpoint':SELECTED_EPOCH,'threshold':0.001,'topk':50})
diag_cfg['trainer'].update({'save_all':True,'pretrain_model':None,'resume_model':False})
DIAG_CONFIG = MONODETR_REPO/'configs/monodetr_a2_epoch130_diagnostic.yaml'
DIAG_CONFIG.write_text(yaml.safe_dump(diag_cfg,sort_keys=False))
PREDICTION_DIR = RUN_DIR/'outputs/data'
shutil.rmtree(PREDICTION_DIR,ignore_errors=True)
DIAG_LOG = run_training_logged([sys.executable,'-u','tools/train_val.py','--config',DIAG_CONFIG,'--evaluate_only'],MONODETR_REPO)
prediction_files=list(PREDICTION_DIR.glob('*.txt'))
if len(prediction_files) != 3769: raise RuntimeError(f'Expected 3769 prediction files, found {len(prediction_files)}')
DIAGNOSTIC_DIR = OUTPUT_ROOT/'nearby_geometry_epoch130'
run([sys.executable,'-u','scripts/audit_product_prediction_geometry.py','--dataset-root',MONODETR_KITTI,'--split-file',SPLIT_DIR/'val.txt','--prediction-dir',PREDICTION_DIR,'--output-dir',DIAGNOSTIC_DIR,'--checkpoint',SELECTED_CHECKPOINT,'--expected-checkpoint-sha256',SELECTED_SHA256,'--expected-images','3769','--score-threshold','0.001','--match-iou-threshold','0.5'],cwd=MOBILE_REPO)
summary=__import__('json').loads((DIAGNOSTIC_DIR/'nearby_geometry_summary.json').read_text())
print(__import__('json').dumps(summary['classes'],indent=2))
display(pd.read_csv(DIAGNOSTIC_DIR/'geometry_summary.csv'))
print('Return:',DIAGNOSTIC_DIR/'nearby_geometry_summary.json')
